## Setting up the Environment

**Load the required modules and packages**

### 1. Environment Setup

In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

# Chrome options
options = webdriver.ChromeOptions()
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

### 2. Load Existing Song Links Database

In [ ]:
try:
    df_links = pd.read_csv("songs.csv")
    song_links = set(df_links["link"].tolist())
except FileNotFoundError:
    print("songs.csv not found. Starting fresh.")
    song_links = set()

In [ ]:
print(f"Total song links in database: {len(song_links)}")

### 3. Scrape Spotify for New Song Links

In [ ]:
def scrape_song_links(driver, song_links):
    driver.get('https://open.spotify.com/artist/06HL4z0CvFAxyc27GXpf02/discography/all')
    driver.implicitly_wait(30)
    time.sleep(60)  # Wait for dynamic content to load
    elements = driver.find_elements("xpath", "//div[@class='hb8C1VAjyUg0VMxrwpixac']/a[@class='eYJgrgW01l7dHKuMJidG']")
    new_links = set()
    for elem in elements:
        try:
            url = elem.get_attribute("href")
            if url and url not in song_links:
                new_links.add(url)
        except StaleElementReferenceException:
            continue
    print(f"New links found: {len(new_links)}")
    song_links = song_links.union(new_links)
    return song_links

In [ ]:
song_links = scrape_song_links(driver, song_links)
print(f"Total song links in database: {len(song_links)}")

In [ ]:
# Save updated links
pd.DataFrame({"link": list(song_links)}).to_csv("songs.csv", index=False, header=True)

### 4. Load Existing Song Details Database

In [ ]:
try:
    df_details = pd.read_csv("songsdetails.csv")
    details_scraped = set(df_details["SongLink"].tolist())
except FileNotFoundError:
    print("songsdetails.csv not found. No previously scraped details.")
    df_details = pd.DataFrame()
    details_scraped = set()

### 5. Scrape Song Details (for new songs)

In [ ]:
# Set implicit wait
driver.implicitly_wait(10)

# Convert song_links to list (if not already)
song_links = list(song_links)

# Initialize dictionary to store information
songsdic = {"SongName": [], "AlbumName": [], "AlbumLink": [], "Year": [], "Runtime": [], "PlayCount": [], "Lyrics": [], "SongLink": []}

# Convert details_scraped to a set for faster lookups
details_scraped = set(details_scraped)

# Define batch size
batchsize = 50
numsongs = len(song_links)

for batchstart in range(0, numsongs, batchsize):
    batchend = min(batchstart + batchsize, numsongs)
    songbatch = song_links[batchstart:batchend]

    for i in songbatch:
        # Skip already-scraped songs
        if i in details_scraped:
            continue  

        driver.get(i)
        time.sleep(10)
        try:
            songname = driver.find_element("xpath", "//div[@class='c55UACltdzzDDQVfoF18']/span[2]/span/h1").get_attribute("textContent").strip()
        except NoSuchElementException:
            songname = "No Name"

        try:
            albumname_elem = driver.find_element("xpath", "//div[@class='JWDnag2Mepdf9QE0cNbg']/span[2]/a")
            albumname = albumname_elem.get_attribute("textContent").strip()
            albumlink = albumname_elem.get_attribute("href")
        except NoSuchElementException:
            albumname = "No Album Name"
            albumlink = "No Album Link"

        try:
            year = driver.find_element("xpath", "//div[@class='JWDnag2Mepdf9QE0cNbg']/span[4]").get_attribute("textContent").strip()
        except NoSuchElementException:
            year = "No Year"

        try:
            runtime = driver.find_element("xpath", "//div[@class='JWDnag2Mepdf9QE0cNbg']/span[6]").get_attribute("textContent").strip()
        except NoSuchElementException:
            runtime = "No Runtime"

        try:
            playcount = driver.find_element("xpath", "//div[@class='JWDnag2Mepdf9QE0cNbg']/span[8]").get_attribute("textContent").strip()
        except NoSuchElementException:
            playcount = "0"

        try:
            lyricselements = driver.find_elements("xpath", "//p[@class='e-91000-text encore-text-body-medium NqaDCx7q6vMeN6tO4Kpf']")
            lyrics = "\n".join([elem.get_attribute("textContent").strip() for elem in lyricselements])
            if not lyrics:
                lyrics = "No Lyrics"
        except Exception:
            lyrics = "No Lyrics"

        # Append scraped data
        songsdic["SongName"].append(songname)
        songsdic["AlbumName"].append(albumname)
        songsdic["AlbumLink"].append(albumlink)
        songsdic["Year"].append(year)
        songsdic["Runtime"].append(runtime)
        songsdic["PlayCount"].append(playcount)
        songsdic["Lyrics"].append(lyrics)
        songsdic["SongLink"].append(i)


In [ ]:
# Save new details
if not df_details.empty:
    df_details = pd.concat([df_details, pd.DataFrame(songsdic)], ignore_index=True)
else:
    df_details = pd.DataFrame(songsdic)

In [ ]:
df_details.to_csv("songsdetails.csv", index=False)

### 6. Data Cleaning and Feature Engineering

In [ ]:
df = df_details.copy()
df.head()

In [ ]:
df.columns

In [ ]:
df.loc[df['PlayCount'] == 'No Play Count', 'PlayCount'] = 0


In [ ]:
df["Year"] = df["Year"].astype(int, errors='ignore')
df[["Minutes", "Seconds"]] = df["Runtime"].str.split(":", expand=True)
df["Minutes"] = pd.to_numeric(df["Minutes"], errors="coerce").fillna(0).astype(int)
df["Seconds"] = pd.to_numeric(df["Seconds"], errors="coerce").fillna(0).astype(int)
df["Duration"] = df["Minutes"] * 60 + df["Seconds"]
df = df.drop(["Minutes", "Seconds", "Runtime"], axis=1)
df["PlayCount"] = df["PlayCount"].astype(str).str.replace(",", "").astype(float).round(0).astype("Int64")
df["Lyrics"] = df["Lyrics"].fillna("").astype(str)
df["SongName"] = df["SongName"].fillna("").astype(str)
df["AlbumName"] = df["AlbumName"].fillna("").astype(str)

In [ ]:
num_duplicates = df.duplicated(subset="Lyrics").sum()
print("Number of duplicate entries based on Lyrics:", num_duplicates)

In [ ]:
# Remove duplicates based on Lyrics
df = df.drop_duplicates(subset="Lyrics", keep="first")

In [ ]:
# Save cleaned data
df.to_csv("cleanedsongs.csv", index=False)

In [ ]:
# Print summary
print(f"{len(df)} unique songs after cleaning and removing duplicates.")

### 7. Data Overview and Example Outputs

In [ ]:
df.head()